In [ ]:
!pip install monai itk einops nibabel

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from monai.networks.nets import MaskedAutoEncoderViT
import monai.transforms as mt
import monai
from pathlib import Path
from tqdm import tqdm
from google.colab import drive


In [ ]:
drive.mount('/content/drive')

class MAEDataSet(monai.data.Dataset):
    def __init__(self, imagesTr, imagesTr_unlabeled):
        train_labeled = sorted(Path(imagesTr).glob("*.mha"))
        train_unlabeled = sorted(Path(imagesTr_unlabeled).glob("*.mha"))
        all_paths = train_labeled + train_unlabeled
        data = [{"image": str(path)} for path in all_paths]

        transforms = mt.Compose([
            mt.LoadImaged(keys=["image"], reader="ITKReader"),
            mt.EnsureChannelFirstd(keys=["image"]),
            mt.Spacingd(keys=["image"], pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
            mt.Orientationd(keys=["image"], axcodes="RAS"),
            mt.NormalizeIntensityd(keys=["image"]),
            mt.RandSpatialCropd(keys=["image"], roi_size=(96, 96, 96), random_size=False),
            mt.RandFlipd(keys=["image"], prob=0.2, spatial_axis=0),
            mt.RandFlipd(keys=["image"], prob=0.2, spatial_axis=1),
            mt.RandFlipd(keys=["image"], prob=0.2, spatial_axis=2),
            mt.RandRotate90d(keys=["image"], prob=0.2, max_k=3),
            mt.RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.2),
            mt.RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.2),
        ])
        super().__init__(data=data, transform=transforms)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

dataset = MAEDataSet(
    "/content/drive/MyDrive/panther/ImagesTr/",
    "/content/drive/MyDrive/panther/ImagesTr_unlabeled/"
)

dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=4)

model = MaskedAutoEncoderViT(
    in_channels=1,
    img_size=(96, 96, 96),
    patch_size=(16, 16, 16),
    masking_ratio=0.75,
    spatial_dims=3,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
epochs = 150
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

best_loss = float('inf')

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}"):
        patch = batch["image"].to(device)
        optimizer.zero_grad()
        output = model(patch)
        reconstructed = output[0]
        mask = output[1]
        B = patch.shape[0]
        original = patch.reshape(B, 1, 6, 16, 6, 16, 6, 16)
        original = original.permute(0, 2, 4, 6, 1, 3, 5, 7)
        original = original.reshape(B, 216, -1)
        loss = F.mse_loss(reconstructed[mask == 1], original[mask == 1])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{epochs} — Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "/content/drive/MyDrive/MAE_models/mae_model_phase1_v1.pth")
        print(f"Model saved (loss: {best_loss:.4f})")

    scheduler.step()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


Epoch 1/150: 100%|██████████| 115/115 [05:18<00:00,  2.77s/it]


Epoch 1/150 — Loss: 1.0552
Model saved (loss: 1.0552)


Epoch 2/150: 100%|██████████| 115/115 [05:34<00:00,  2.91s/it]


Epoch 2/150 — Loss: 0.9249
Model saved (loss: 0.9249)


...

Epoch 148/150 — Loss: 0.2916


Epoch 149/150: 100%|██████████| 115/115 [05:31<00:00,  2.89s/it]


Epoch 149/150 — Loss: 0.2958


Epoch 150/150: 100%|██████████| 115/115 [05:32<00:00,  2.89s/it]

Epoch 150/150 — Loss: 0.2982
